In [4]:
from astropy.table import Table
import redrock

import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt


### read in master table from Jay (BGS bright, faint, dwarf with fainter than -17 in r)

In [69]:
fn = '/pscratch/sd/n/nravi/BTFR/DR2_BGS_dwarfs.fits'
master_table = Table.read(fn)
master_table[:5]

TARGETID,RA,DEC,Z,ABSMAG01_SDSS_U,ABSMAG01_SDSS_G,ABSMAG01_SDSS_R,ABSMAG01_SDSS_Z,LOGMSTAR,SFR,HALPHA_EW,BGS_TARGET
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64
3.962781115298984e+16,135.19570846244682,0.9722796343462936,0.03359060677168832,-14.30962085723877,-15.21914005279541,-15.53191089630127,-15.60239028930664,7.549510955810547,0.003126659197732806,30.79812240600586,131074
3.962781115298867e+16,135.13294145851197,0.9763590204550645,0.01824851078642592,-13.199661254882812,-14.298686027526855,-14.61616039276123,-14.689810752868652,7.176872253417969,0.000152943393914029,26.085153579711914,131074
3.962780511738573e+16,135.37699112704027,0.6318663033977093,0.07037077969344004,-15.483592987060547,-16.30772590637207,-16.64521026611328,-16.868022918701172,8.275031089782715,0.0015191471902653575,79.33531188964844,65537
3.9627799065003496e+16,134.56939198464678,0.4396445519128736,0.01132122340830344,-12.714864730834961,-13.717902183532715,-14.065450668334961,-14.174965858459473,6.925713539123535,2.098329510147323e-08,8.466760635375977,131074
3.962779906500679e+16,134.73505875622243,0.4249612190235197,0.040493346369137005,-15.05160140991211,-16.061798095703125,-16.4205265045166,-16.585397720336914,8.016134262084961,8.796555484025248e-09,21.854976654052734,131074


In [70]:
master_table['TARGETID'] = master_table['TARGETID'].astype(int)
master_table[:5]

TARGETID,RA,DEC,Z,ABSMAG01_SDSS_U,ABSMAG01_SDSS_G,ABSMAG01_SDSS_R,ABSMAG01_SDSS_Z,LOGMSTAR,SFR,HALPHA_EW,BGS_TARGET
int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64
39627811152989840,135.19570846244682,0.9722796343462936,0.03359060677168832,-14.30962085723877,-15.21914005279541,-15.53191089630127,-15.60239028930664,7.549510955810547,0.003126659197732806,30.79812240600586,131074
39627811152988672,135.13294145851197,0.9763590204550645,0.01824851078642592,-13.199661254882812,-14.298686027526855,-14.61616039276123,-14.689810752868652,7.176872253417969,0.000152943393914029,26.085153579711914,131074
39627805117385728,135.37699112704027,0.6318663033977093,0.07037077969344004,-15.483592987060547,-16.30772590637207,-16.64521026611328,-16.868022918701172,8.275031089782715,0.0015191471902653575,79.33531188964844,65537
39627799065003496,134.56939198464678,0.4396445519128736,0.01132122340830344,-12.714864730834961,-13.717902183532715,-14.065450668334961,-14.174965858459473,6.925713539123535,2.098329510147323e-08,8.466760635375977,131074
39627799065006792,134.73505875622243,0.4249612190235197,0.040493346369137005,-15.05160140991211,-16.061798095703125,-16.4205265045166,-16.585397720336914,8.016134262084961,8.796555484025248e-09,21.854976654052734,131074


In [71]:
#dict
master_dict = {}
for i in range(len(master_table)):

    master_dict[master_table['TARGETID'][i]] = i

### read in julia's morph table

In [72]:
# read in julia's morph
morph_table = Table.read('/pscratch/sd/n/nravi/BTFR/dwarf_gals/Dwarf_Irregulars.csv')
morph_table[:5]

targetid,Predicted_Label,Predicted_Type
int64,int64,str9
2389545307865089,0,Other
2389551339274240,0,Other
2389551339274241,0,Other
2389557370683393,0,Other
2389557383266309,0,Other


In [73]:
# add col to master_table for irr class
master_table['Irr'] = np.ones(len(master_table), dtype=int)*-1
for i in range(len(morph_table)):

    idx = master_dict[morph_table['targetid'][i]]
    master_table['Irr'][idx] = morph_table['Predicted_Label'][i]

### keep visible targets for 2026B
1. RA > 310 degrees or RA < 135 degrees (RA>345 or RA<100)
2. -119 < dec < 61 (-89 < dec < 61 for alt above 30 deg)

In [76]:
visible = master_table[np.logical_and(np.logical_or(master_table['RA'] < 100, master_table['RA'] > 345), np.logical_and(master_table['DEC'] < 61, master_table['DEC'] > -89))]
len(visible)

42703

### redshift cut
to match ALFALFA (and keep nearest objects), z < 0.05

In [78]:
visible_near = visible[visible['Z'] < 0.05]
len(visible_near)

31600

### keep brightest objects
Mr < -15

In [79]:
visible_near_bright = visible_near[visible_near['ABSMAG01_SDSS_R'] < -15]
len(visible_near_bright)

24494

In [82]:
visible_near_bright.write('Magellan_2026B_ra345100_dec-8931_z0p05_Mr-15.fits', format='fits')

### check how many have already been classified

In [84]:
visible_near_bright[visible_near_bright['Irr'] == -1]

TARGETID,RA,DEC,Z,ABSMAG01_SDSS_U,ABSMAG01_SDSS_G,ABSMAG01_SDSS_R,ABSMAG01_SDSS_Z,LOGMSTAR,SFR,HALPHA_EW,BGS_TARGET,Irr
int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64
39633314088224656,99.56795050929773,54.473985285133786,0.04343839337721014,-15.546764373779297,-16.453800201416016,-16.8712100982666,-17.044300079345703,8.407883644104004,7.809236668474375e-16,42.54764938354492,514,-1
39633321096905704,99.40687460001632,55.04023356719571,0.043867910030580756,-14.836222648620605,-15.666781425476074,-16.029178619384766,-16.09145736694336,7.918125629425049,0.0037914582062512636,36.67704772949219,514,-1
39633317598855640,98.88565526721487,54.77550226910008,0.04643928397530949,-15.670310020446777,-16.307899475097656,-16.497207641601562,-16.402202606201172,7.4413862228393555,0.003001233795657754,167.49142456054688,514,-1
39633317598855440,98.86969988846631,54.7614715667157,0.04565840191377814,-15.422883987426758,-16.366920471191406,-16.777549743652344,-16.957853317260742,8.287899017333984,5.1897468961170645e-11,3.392954111099243,514,-1
39633324557206368,97.94965881205779,55.190112142598316,0.044297158540045276,-15.386693000793457,-16.137083053588867,-16.5494384765625,-16.79900360107422,8.368792533874512,0.0019596314523369074,20.977542877197266,514,-1
39633324557206048,97.91641376932802,55.28405749340821,0.025803425638446613,-13.982528686523438,-15.218293190002441,-15.738493919372559,-15.874903678894043,8.001702308654785,8.621139357956054e-10,18.50542640686035,514,-1
39633321084325192,98.31777342907382,55.115928511699124,0.02884305042271551,-14.178070068359375,-15.068110466003418,-15.537923812866211,-15.817173957824707,8.016911506652832,0.0021238040644675493,43.67566680908203,514,-1
39633324557209408,98.25536668628098,55.22270880479468,0.040701076211527396,-15.685564994812012,-16.511768341064453,-16.963394165039062,-17.266319274902344,8.67279052734375,1.2313051911405637e-07,26.540876388549805,514,-1
39633324561402320,98.52659343859715,55.20192135099961,0.04375364987132804,-16.032577514648438,-16.655967712402344,-16.774925231933594,-16.762908935546875,8.42387580871582,0.007690548896789551,203.679443359375,514,-1


In [94]:
visible_near_bright[visible_near_bright['DEC'] < -2].write('Magellan_2026B_ra345100_dec-8931_z0p05_Mr-15_priority.fits', format='fits')

In [96]:
visible_near_bright[visible_near_bright['DEC'] >= -2].write('Magellan_2026B_ra345100_dec-8931_z0p05_Mr-15_low_priority.fits', format='fits')

In [98]:
visible_near_bright[visible_near_bright['DEC'] < -2][300:310]

TARGETID,RA,DEC,Z,ABSMAG01_SDSS_U,ABSMAG01_SDSS_G,ABSMAG01_SDSS_R,ABSMAG01_SDSS_Z,LOGMSTAR,SFR,HALPHA_EW,BGS_TARGET,Irr
int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64
39627676092206784,1.3658144562767032,-4.511588379020368,0.03639921122966487,-15.004794120788574,-16.218978881835938,-16.681896209716797,-16.882558822631836,8.30964183807373,1.0010290196760252e-07,16.705190658569336,131074,-1
39627694178047752,1.4343168136071813,-3.6322655903594394,0.020093671134922865,-14.568513870239258,-15.324019432067871,-15.635618209838867,-15.756169319152832,7.425498008728027,6.217034109745612e-10,49.49300765991211,131074,-1
39627688138250864,0.9501411712214196,-4.009093890428035,0.04884904450275619,-14.675985336303711,-15.425058364868164,-15.837210655212402,-16.051971435546875,8.096778869628906,0.002869797870516777,15.71810245513916,65545,-1
39627694178047216,1.417162176402041,-3.7702889268880644,0.02231354018545352,-14.181861877441406,-14.853997230529785,-15.166534423828125,-15.281301498413086,7.583858966827393,2.766075563043202e-12,66.75721740722656,131074,-1
39627688142442768,1.1525777431317654,-3.8926512982812835,0.021504810074986266,-15.06937313079834,-16.142147064208984,-16.9290828704834,-17.422332763671875,8.794422149658203,5.766423701846497e-18,0.1271033138036728,131074,-1
39627694178043256,1.2918918326496136,-3.8476512788425,0.021162158297380013,-15.189168930053711,-15.869510650634766,-16.236730575561523,-16.42923927307129,8.044332504272461,2.778517860235752e-09,24.921913146972656,131074,-1
39627676092204520,1.2961663634347818,-4.455696723602669,0.0384909692809839,-14.850970268249512,-16.144561767578125,-16.77246856689453,-17.13451385498047,8.574131965637207,2.6902749472590415e-10,5.472781181335449,131074,-1
39627682106835896,0.7751502165833356,-4.196435154040907,0.021399411112371196,-14.151040077209473,-15.055153846740723,-15.417163848876953,-15.568294525146484,7.493027210235596,4.9727319484382804e-11,10.941064834594727,131074,-1
39627694178043080,1.286583093634324,-3.8077679170701693,0.02582045966684309,-14.934104919433594,-16.15252113342285,-16.646163940429688,-16.838966369628906,8.538434028625488,1.883286637749393e-09,3.859722852706909,131074,-1
